This file contains our Scikit-learn unsupervised models, which can be compared to find the most effective model and to aid in writing the research manuscript. The Clean_and_EDA.py must be ran inside this file so that the models have access to the data. There is plots, other figures, and accuracy scores so we can compare and validate our models. Model 1 is the isolated forest model. Contributions by Brendan Daly.

In [ ]:
# Load in libraries
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.ensemble import IsolationForest

In [ ]:
# Run the cleaning script once to obtain the data here for the models to use!
%run Clean_Data.py

In [ ]:
# Select features to use in the model
# List of all features, most features should not be used in each model
features_list = ['Date','Date Time','Lactation Days','Programmed Rest Feed','Rest Feed','Day Production','Day Production Dev','ISK','Milkings','Refusals','Failures','Milking Avg','Refusals Avg','Failures Avg','Fat Indication','Protein Indication','Lactose Indication','Address','Failure','Milk Yield','Milk Yield Expected','Conductivity LF','Conductivity RF','Conductivity LR','Conductivity RR','Color Code RF','Color Code LR','Color Code RR','Milk Temperature','Device Name','Visit Result','Milk Speed','Milk Speed Max','Milking Interval (min)','Box Time (min)','Treat Time (min)','Dead Milk Time LF (min)','Dead Milk Time RF (min)','Dead Milk Time LR (min)','Dead Milk Time RR (min)','Milk Time LF (min)','Milk Time RF (min)','Milk Time LR (min)','Milk Time RR (min)','Attachments LF','Attachments RF','Attachments LR','Attachments RR','Milk Destination','Routing Direction','Bottle Num','Lactation Number','Calving Date','Lactation Production','Predicted Production 305d','Milk Separated','Amount Total','Rest Feed Total','Fat %','Protein %','Lactose %','Event Type','Lactation Section','Lactation Day','Event Description']

# Isolated Forest Model Features
iso_forest_features = [
    'Lactation Days',
    'Day Production',
    'Day Production Dev',
    'Milkings',
    'Refusals',
    'Failures',
    'Milk Yield',
    'Milk Yield Expected',
    'Conductivity LF',
    'Conductivity RF',
    'Conductivity LR',
    'Conductivity RR',
    'Milk Temperature',
    'Milk Speed',
    'Milk Speed Max',
    'Milking Interval (min)',
    'Fat %',
    'Protein %',
    'Lactose %',
    'Lactation Number'
]


# Select Isolated Forest Model Features from Each Sheet
iso_forest_features_dict = {}

for file_name, tables in Robot_milking_data_renamed.items():
    iso_forest_features_dict[file_name] = {}

    for table_name, df in tables.items():
        missing = [
            feature for feature in iso_forest_features
            if feature not in df.columns
        ]

        if missing:
            print(f"{file_name} | {table_name}: missing {missing}")

        available = [
            feature for feature in iso_forest_features
            if feature in df.columns
        ]

        iso_forest_features_dict[file_name][table_name] = (
            df[available].copy()
        )

first_file = next(iter(iso_forest_features_dict))
first_table = next(iter(iso_forest_features_dict[first_file]))
df_features = iso_forest_features_dict[first_file][first_table]

# (Yikes, lets just clean the data fully this time...)
print(df_features.shape)
print(df_features.head())

In [ ]:
# Parameters (Experiment with the model by changing these!)
n_estimators = 100 # number of trees
contamination = 0.01 # expected proportion of anomalies
sample_size = 256 # number of samples used to train each tree

In [ ]:
# Train Isolated Forest Model
iso_forest = IsolationForest(n_estimators=n_estimators,
                             contamination=contamination,
                             max_samples=sample_size,
                             random_state=123) # Reproducibility
iso_forest.fit(iso_forest_features)

In [ ]:
# Calculate anomaly scores and classify anomalies
Robot_milking_data_renamed = Robot_milking_data_renamed.loc[iso_forest_features.index].copy()
Robot_milking_data_renamed['anomaly_score'] = iso_forest.decision_function(iso_forest_features)
Robot_milking_data_renamed['anomaly'] = iso_forest.predict(iso_forest_features)
Robot_milking_data_renamed['anomaly'].value_counts()

In [ ]:
# Visualization of the results
plt.figure(figsize=(10, 5))

# Plot normal instances
normal = Robot_milking_data_renamed[Robot_milking_data_renamed['anomaly'] == 1]
plt.scatter(normal.index, normal['anomaly_score'], label='Normal')

# Plot anomalies
anomalies = Robot_milking_data_renamed[Robot_milking_data_renamed['anomaly'] == -1]
plt.scatter(anomalies.index, anomalies['anomaly_score'], label='Anomaly')
plt.xlabel("Instance")
plt.ylabel("Anomaly Score")
plt.legend()
plt.show()